### Imports

In [ ]:
import torch
from models.transformer import TransformerLanguageModel
from utils import char_tokenizer, word_tokenizer

### Configuration

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# -------------------------------------------------
# Configuration
# -------------------------------------------------
TOKEN_LEVEL = "char"   # choose: "char" or "word"
TEXT_PATH = "../data/sherlock.txt"
MODEL_DIR = ".."
GENERATE_LENGTH = 200
START_TEXT = "To Sherlock Holmes she is always the woman"

### Load Dataset

In [ ]:
# -------------------------------------------------
# Load text
# -------------------------------------------------
text = open(TEXT_PATH).read()

### Tokenization

In [ ]:
# -------------------------------------------------
# Tokenization
# -------------------------------------------------
if TOKEN_LEVEL == "char":
    encoded, stoi, itos = char_tokenizer(text)
    model_path = f"{MODEL_DIR}/char_transformer.pt"

elif TOKEN_LEVEL == "word":
    encoded, stoi, itos = word_tokenizer(text)
    model_path = f"{MODEL_DIR}/word_transformer.pt"

else:
    raise ValueError("TOKEN_LEVEL must be 'char' or 'word'")

### Load Model

In [ ]:
# -------------------------------------------------
# Load model
# -------------------------------------------------
model = TransformerLanguageModel(len(stoi)).to(device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

### Text Generation Function

In [ ]:
# -------------------------------------------------
# Text generation function
# -------------------------------------------------
def generate_text(start_text, length=200):

    if TOKEN_LEVEL == "char":
        tokens = [stoi[c] for c in start_text]

    else:
        tokens = [stoi[w] for w in start_text.split() if w in stoi]

    context = torch.tensor([tokens]).to(device)

    for _ in range(length):

        logits = model(context)

        probs = torch.softmax(logits[0, -1], dim=0)

        next_token = torch.multinomial(probs, 1).item()

        context = torch.cat(
            [context, torch.tensor([[next_token]]).to(device)],
            dim=1
        )

    generated_tokens = [itos[i] for i in context[0].tolist()]

    if TOKEN_LEVEL == "char":
        generated = "".join(generated_tokens)
    else:
        generated = " ".join(generated_tokens)

    return generated

### Generate Example

In [ ]:
# -------------------------------------------------
# Run generation
# -------------------------------------------------
output = generate_text(START_TEXT, GENERATE_LENGTH)

print(f"\nTokenization Level: {TOKEN_LEVEL}")
print("\nGenerated Text:\n")
print(output)